# 03 · EDA: understand the data

Questions this notebook answers before any modeling:
1. **Do we have enough data?** How many games and team-game rows can we train on?
2. **What does the target look like?** Distribution of team points, totals and margins.
3. **Is the game stable over time?** If scoring changes by era, old seasons mislead the model.
4. **What obvious effects exist?** Home field, neutral sites, FCS opponents, quarters.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from canes_cfb.paths import RAW

games = pd.read_parquet(RAW / "games.parquet")
done = games[games.completed & (games.season <= 2025)].copy()
done["total"] = done.home_points + done.away_points
done["margin"] = done.home_points - done.away_points
fbs = done[done.home_fbs & done.away_fbs]
print(f"completed 2015–2025: {len(done):,} | FBS vs FBS: {len(fbs):,}")

## 1. Do we have enough data?

The first model predicts **each team's points**, so every game gives two training rows.
Games against FCS teams are excluded from training because they're lopsided
(average margin ~31 points) and not what we bet on.

In [ ]:
volume = pd.DataFrame(
    {
        "games": done.groupby("season").size(),
        "fbs_vs_fbs": fbs.groupby("season").size(),
        "vs_fcs": done[done.home_fbs ^ done.away_fbs].groupby("season").size(),
    }
)
volume["team_rows"] = volume.fbs_vs_fbs * 2
volume.loc["total"] = volume.sum()
volume

**Answer:** ~8,300 FBS-vs-FBS games (2015–2025) → ~16,600 team-game rows.
That's plenty for linear models and enough for gradient boosting with modest depth and
regularization. It isn't big-data territory, so feature quality will matter more than
model complexity. That supports putting the effort into feature engineering.

Split (see `docs/pipeline.md`): train 2015–2023, validation 2024, test 2025.

## 2. What does the target look like?

In [ ]:
team_points = pd.concat([fbs.home_points, fbs.away_points])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
team_points.plot.hist(bins=range(0, 80, 2), ax=axes[0], title="Team points")
fbs.total.plot.hist(bins=range(0, 130, 3), ax=axes[1], title="Game total")
fbs.margin.clip(-60, 60).plot.hist(bins=range(-60, 61), ax=axes[2], title="Home margin")
plt.tight_layout()
print(f"team points: mean {team_points.mean():.1f}, sd {team_points.std():.1f}")
print(f"game total:  mean {fbs.total.mean():.1f}, sd {fbs.total.std():.1f}")

Team points average ~27.7 with SD ~13.9, which is wide. A model that always guesses 27.7
misses by ~11 points on average. That's the naive baseline to beat.

**Key numbers:** margins of exactly 3 and 7 happen far more often than their neighbors
(field goal and touchdown). A normal distribution around a predicted margin ignores this,
which matters when turning a predicted margin into a cover probability.

In [ ]:
(fbs.margin.abs().value_counts(normalize=True).sort_index().loc[1:21] * 100).plot.bar(
    figsize=(10, 3), title="% of games decided by exactly N points"
);

## 3. Is scoring stable over time?

In [ ]:
fbs.groupby("season").total.mean().plot(
    marker="o", figsize=(8, 3), title="Avg game total by season"
);

**No.** The average total fell from ~57 (2015–2016) to ~52 (2025). The 2023 clock rule
change (the clock keeps running after first downs) cut plays per game.

Implications for features:
- Raw "points per game" from 2016 isn't comparable to 2025. Features should be relative
  to the **season average** (or adjusted by era), not raw points.
- Recent seasons should weigh more (sample weights or a shorter training window). This
  is one of the things to test in CV.

## 4. Home field and neutral sites

In [ ]:
print(fbs.groupby("neutral_site").margin.agg(["mean", "count"]).round(2))

Home teams win by ~4.3 points on average at their own stadium, and ~1 at neutral sites.
`is_home` and `neutral_site` are required features. Home advantage may also vary by team
and stadium (a later feature to test).

## 5. Scoring by quarter

In [ ]:
per_q = pd.Series({f"Q{i}": (fbs[f"home_q{i}"] + fbs[f"away_q{i}"]).mean() for i in range(1, 5)})
per_q["OT games %"] = ((fbs.home_ot + fbs.away_ot) > 0).mean() * 100
per_q.round(1)

Q2 is the highest-scoring quarter (~16 points, end-of-half drives) and Q1 is the lowest
(~12). Quarter models can't just divide the full-game prediction by 4. Each quarter has
its own share, and that share probably depends on the teams (fast starters, strong
second halves). That's a feature idea for the quarter and half models.

## Summary

| Question | Answer |
|---|---|
| Enough data? | Yes: ~16,600 team-game rows (2015–2025). Good for linear models and regularized boosting. |
| Target | Team points, mean ~27.7, SD ~13.9. Naive baseline error is ~11 points. |
| Stable over time? | No: scoring is down ~5 points since 2015. Normalize by season; weigh recent years. |
| Home field | +4.3 at home, +1 neutral. |
| FCS games | Exclude from training targets (avg margin ~31). |
| Quarters | Uneven (Q2 highest). Model periods explicitly. |

Next: `04_feature_engineering`.